In [20]:
import pandas as pd
import sqlite3

In [21]:
conn = sqlite3.connect("dataset/rideshare.db")

In [22]:
df = pd.read_sql_query("SELECT status FROM trips LIMIT 5", conn)

In [23]:
df

,status
0,completed
1,cancelled
2,completed
3,completed
4,completed


In [24]:
conn.close()

In [25]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

In [26]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

In [27]:
print("Models available:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

Models available:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-

In [28]:
def get_database_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    conn.close()
    schema = "\n\n".join([table[0] for table in tables if table[0] is not None])
    return schema

In [29]:
db_schema = get_database_schema("dataset/rideshare.db")

In [33]:
def run_sql_query(query: str) -> str:
    try:
        conn = sqlite3.connect("dataset/rideshare.db")
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df.to_string(index=False) 
    except Exception as e:
        return f"SQL Error: {str(e)}"

In [47]:
def agent(user_question):

    system_prompt = f"""
    You are an expert SQL Assistant. Your job is to convert natural language questions into SQL queries based on the following SQLite database schema:
    SCHEMA:
    {db_schema}
    RULES:
    1. If the question can be answered using the tables in the schema, return ONLY the raw SQL query. Do not include markdown formatting (like ```sql), do not include explanations. Just the query.
    2. If the question is general English, unrelated to the schema, or impossible to answer with the provided tables, you must reply EXACTLY with:
       "None"
    """
    system_prompt = f"""
    You are a brilliant autonomous SQL Agent. You have access to a database with the following schema:
    SCHEMA:
    {db_schema}
    RULES:
    1. When asked a question, you must write a SQL query and use the `run_sql_query` tool to execute it.
    2. If the tool returns a "SQL Error", you must read the error, fix your SQL, and try the tool again.
    3. Once you successfully get the data from the tool, synthesize a natural, friendly final answer in English for the user.
    4. If the question is general English, unrelated to the schema, or impossible to answer with the provided tables, you must reply EXACTLY with:
       "I'm sorry. Your question is invalid or unrelated to the database. I don't know the answer."
    """

    agent_model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        tools=[run_sql_query], 
        system_instruction=system_prompt
    )

    agent = agent_model.start_chat(enable_automatic_function_calling=True)
    response = agent.send_message(user_question)
    return response.text

In [48]:
print(agent("How many total users are there in the database?"))

There are a total of 2000 users in the database.


In [49]:
print(agent("List the names of the top 3 drivers based on their rating."))

The top 3 drivers based on their rating are Cynthia Cook, Patricia Bailey, and Brandon Reyes.


In [50]:
print(agent("What is the capital of France?"))

I'm sorry. Your question is invalid or unrelated to the database. I don't know the answer.


In [53]:
print(agent("How are you?"))

I'm sorry. Your question is invalid or unrelated to the database. I don't know the answer.
